## Stage 0 — World and LLM Setup

Loads the PR2-apartment world fixture and creates the LLM.

`symbol_type` scopes entity grounding and world serialisation to physical bodies.
It defaults to `Symbol` (all instances) inside `llmr` — passing `Body` gives
a tighter, physical-body-only scope. It is **never** imported inside the library itself.


In [1]:
from uniworld import load_pr2_apartment_world
from semantic_digital_twin.world_description.world_entity import Body

world, pr2, context = load_pr2_apartment_world()

# Pass Body for physical-body-scoped grounding (tighter than the Symbol default).
symbol_type = Body

print(f"symbol_type : {symbol_type}")
print(f"World loaded.  Scene bodies in SymbolGraph:")
from krrood.symbol_graph.symbol_graph import SymbolGraph
for inst in SymbolGraph().get_instances_of_type(Body):
    from llmr.bridge.world_reader import symbol_display_name
    print(f"  {symbol_display_name(inst)}")

INSTRUCTION = "pick up the milk from the table"
print(f"\nInstruction: {INSTRUCTION!r}")


`polytope` failed to import `cvxopt.glpk`.
will use `scipy.optimize.linprog`
Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


groundable_type : <class 'semantic_digital_twin.world_description.world_entity.Body'>
World loaded.  Scene bodies in SymbolGraph:
  base_link
  base_footprint
  base_bellow_link
  base_laser_link
  fl_caster_rotation_link
  fl_caster_l_wheel_link
  fl_caster_r_wheel_link
  fr_caster_rotation_link
  fr_caster_l_wheel_link
  fr_caster_r_wheel_link
  bl_caster_rotation_link
  bl_caster_l_wheel_link
  bl_caster_r_wheel_link
  br_caster_rotation_link
  br_caster_l_wheel_link
  br_caster_r_wheel_link
  torso_lift_link
  l_torso_lift_side_plate_link
  r_torso_lift_side_plate_link
  torso_lift_motor_screw_link
  imu_link
  head_pan_link
  head_tilt_link
  head_plate_frame
  sensor_mount_link
  high_def_frame
  high_def_optical_frame
  double_stereo_link
  wide_stereo_link
  wide_stereo_optical_frame
  wide_stereo_l_stereo_camera_frame
  wide_stereo_l_stereo_camera_optical_frame
  wide_stereo_r_stereo_camera_frame
  wide_stereo_r_stereo_camera_optical_frame
  narrow_stereo_link
  narrow_stereo_

In [2]:
import rclpy
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher

rclpy.init()
_ros_node = rclpy.create_node('semantic_digital_twin')
import threading
_ros_thread = threading.Thread(target=rclpy.spin, args=(_ros_node,), daemon=True)
_ros_thread.start()

In [3]:
_tf_publisher = TFPublisher(_world=world, node=_ros_node)
_viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)
print("ROS2 publishers started")

ROS2 publishers started


In [ ]:
# import os
# os.environ['OPENAI_API_KEY']="YOUR_OPENAI_API_KEY"

In [4]:
# ── LLM ───────────────────────────────────────────────────────────────────────
# Swap provider/model freely — everything downstream accepts any BaseChatModel.
from dotenv import load_dotenv
from llmr.reasoning.llm_provider import make_llm, LLMProvider

load_dotenv("../.env")

llm = make_llm(LLMProvider.OPENAI, model="gpt-4o-mini", temperature=0.0)
# llm = make_llm(LLMProvider.OLLAMA, model="qwen3.5:9b")
print(f"LLM ready: {llm.model}")

LLM ready: qwen3.5:9b


## Stage 1 — Discover Action Classes

`discover_action_classes()` (from `llmr.pycram`) loads all modules under
`pycram.robot_plans.actions` once, then uses krrood's `recursive_subclasses(ActionDescription)`
to collect every concrete, non-abstract dataclass action class.  All pycram access is
funnelled through `pycram` — no direct pycram import elsewhere in llmr.


In [5]:
from llmr.pycram import discover_action_classes

action_registry = discover_action_classes()

print(f"Discovered {len(action_registry)} action classes:\n")
for name, cls in sorted(action_registry.items()):
    print(f"  {name:30s}  {cls.__module__}.{cls.__qualname__}")


Discovered 24 action classes:

  CarryAction                     pycram.robot_plans.actions.core.robot_body.CarryAction
  CloseAction                     pycram.robot_plans.actions.core.container.CloseAction
  CuttingAction                   pycram.robot_plans.actions.composite.tool_based.CuttingAction
  DetectAction                    pycram.robot_plans.actions.core.misc.DetectAction
  EfficientTransportAction        pycram.robot_plans.actions.composite.transporting.EfficientTransportAction
  FaceAtAction                    pycram.robot_plans.actions.composite.facing.FaceAtAction
  FollowToolCenterPointPathAction  pycram.robot_plans.actions.core.robot_body.FollowToolCenterPointPathAction
  GraspingAction                  pycram.robot_plans.actions.core.pick_up.GraspingAction
  LookAtAction                    pycram.robot_plans.actions.core.navigation.LookAtAction
  MixingAction                    pycram.robot_plans.actions.composite.tool_based.MixingAction
  MoveAndPickUpAction       

## Stage 2 — LLM Classifies Action Type

`infer_action_class()` is the public classification entry point. Internally it builds a schema-aware action catalog from `ActionFieldIntrospector`, asks the LLM for an `ActionClassificationResult`, and returns the matching Python action class.


In [6]:
from llmr.bridge.introspect import ActionFieldIntrospector

# Preview the action catalog at a high level before calling the public classifier.
# The actual classifier prompt is built inside llmr.reasoning.slot_filler.infer_action_class().
catalog_introspector = ActionFieldIntrospector()

print("=== Available action classes (schema preview) ===")
for name in sorted(action_registry):
    cls = action_registry[name]
    try:
        schema = catalog_introspector.introspect(cls)
        fields = ", ".join(f"{f.name}:{getattr(f.raw_type, '__name__', f.raw_type)}" for f in schema.fields)
    except Exception:
        fields = "<schema unavailable>"
    print(f"  - {name}: {fields}")

print()
print(f"Instruction: {INSTRUCTION!r}")


=== Available action classes (schema preview) ===
  - CarryAction: arm:Arms, align:bool, tip_link:str, tip_axis:AxisIdentifier, root_link:str, root_axis:AxisIdentifier
  - CloseAction: object_designator:Body, arm:Arms, grasping_prepose_distance:float
  - CuttingAction: object_:Body, tool:SemanticAnnotation, arm:Arms, technique:str, slice_thickness:float
  - DetectAction: technique:DetectionTechnique, state:DetectionState, object_sem_annotation:SemanticAnnotation, region:Region
  - EfficientTransportAction: object_designator:Body, target_location:Pose
  - FaceAtAction: pose:Pose, keep_joint_states:bool
  - FollowToolCenterPointPathAction: target_locations:PoseTrajectory, arm:Arms
  - GraspingAction: object_designator:Body, arm:Arms, grasp_description:GraspDescription
  - LookAtAction: target:Pose, camera:Camera
  - MixingAction: object_:Body, tool:SemanticAnnotation, arm:Arms, technique:str
  - MoveAndPickUpAction: standing_position:Pose, object_designator:Body, arm:Arms, grasp_descript

In [7]:
from llmr.reasoning.slot_filler import infer_action_class

classification = infer_action_class(
    instruction=INSTRUCTION,
    llm=llm,
    action_registry=action_registry,
)
assert classification is not None, "LLM could not classify the instruction"

# infer_action_class returns the raw ActionClassificationResult; callers resolve the
# concrete class via the same registry the classifier saw.
action_cls = action_registry.get(classification.action_type)

print("=== LLM classification response ===")
print(f"Classification: {classification}")
print(f"Resolved class: {action_cls}")
assert action_cls is not None, "classified action_type not found in registry"
assert action_cls.__name__ == "PickUpAction"


=== LLM classification response ===
Classification: action_type='PickUpAction' confidence=1.0 reasoning=''
Resolved class: <class 'pycram.robot_plans.actions.core.pick_up.PickUpAction'>


## Stage 3 — Build a Required-Field Match

For the notebook we build the `Match` explicitly from the public `ActionFieldIntrospector` schema: every required, prompt-settable action field is set to `...` (Ellipsis), which means the backend must resolve it.

The higher-level `plan_from_instruction()` API does this internally.


In [ ]:
from krrood.entity_query_language.query.match import Match
from llmr.bridge.introspect import FieldKind, ActionFieldIntrospector

introspector = ActionFieldIntrospector()
action_schema = introspector.introspect(action_cls)
field_specs = {f.name: f for f in action_schema.fields}

required_fields = [
    f.name
    for f in action_schema.fields
    if not f.is_optional and not f.name.startswith("_")
]

def underspecified_match_from_schema(action_cls, schema):
    def value_for_field(field):
        if field.kind != FieldKind.COMPLEX:
            return ...
        nested = Match(field.raw_type)
        nested_kwargs = {
            sub.name: value_for_field(sub)
            for sub in field.sub_fields
            if not sub.is_optional and not sub.name.startswith("_")
        }
        return nested(**nested_kwargs) if nested_kwargs else nested

    kwargs = {
        field.name: value_for_field(field)
        for field in schema.fields
        if not field.is_optional and not field.name.startswith("_")
    }
    match = Match(action_cls)
    return match(**kwargs) if kwargs else match

print(f"Required prompt-settable fields on {action_cls.__name__}:")
for name in required_fields:
    fspec = field_specs[name]
    type_name = fspec.raw_type.__name__ if isinstance(fspec.raw_type, type) else str(fspec.raw_type)
    print(f"  {name:30s}  kind={fspec.kind.value:10s}  type={type_name}")

# Build the required-field Match shape used by llmr.plan_from_instruction().
# Required complex fields become nested Match(...) objects so KRROOD owns recursive construction.
match = underspecified_match_from_schema(action_cls, action_schema)

print()
print(f"Match object : {match}")
print(f"Matches with variables ({len(list(match.matches_with_variables))}):")
for attr in match.matches_with_variables:
    print(f"  field={attr.name_from_variable_access_path:25s}  "
          f"value={attr.assigned_variable._value_!r:10}  "
          f"type={attr.assigned_variable._type_}")


In [ ]:
required_fields

## Stage 3b — Introspect Action Schema with ActionFieldIntrospector

`ActionFieldIntrospector.introspect()` reads the pycram action dataclass and classifies
each field into a `FieldKind`.  This drives the **dynamic LLM prompt**.

| FieldKind | How resolved in `_evaluate` |
|-----------|------------|
| `ENTITY`   | EntityGrounding → Symbol instance (includes Manipulator, Camera) |
| `POSE`     | EntityGrounding → body.global_pose |
| `ENUM`     | string → enum member coercion |
| `COMPLEX`  | represented as nested `Match(...)`; KRROOD constructs the dataclass from resolved leaves |
| `PRIMITIVE` | string coerced to bool/int/float/str |

Note: `_evaluate` classifies slots using krrood's **already-resolved** field types
(`attr_match.assigned_variable._type_`) — no full action-class re-introspection.


In [ ]:
from llmr.bridge.introspect import FieldKind

# Reuse the schema from Stage 3.
print(f"=== ActionSchema for {action_schema.action_type} ===")
print(f"Docstring: {action_schema.docstring[:120]}...")
print(f"Fields ({len(action_schema.fields)}):")

for fspec in action_schema.fields:
    type_name = fspec.raw_type.__name__ if isinstance(fspec.raw_type, type) else str(fspec.raw_type)
    opt_str = " [optional]" if fspec.is_optional else " [required]"
    print(f"{fspec.name:30s}  kind={fspec.kind.value:10s}  type={type_name}{opt_str}")
    if fspec.docstring:
        print(f"    doc: {fspec.docstring[:100]}")
    if fspec.enum_members:
        print(f"    allowed values: {', '.join(fspec.enum_members)}")
    if fspec.sub_fields:
        print("    sub-fields:")
        for sub in fspec.sub_fields:
            sub_type = sub.raw_type.__name__ if isinstance(sub.raw_type, type) else str(sub.raw_type)
            members_note = f"  allowed: {', '.join(sub.enum_members)}" if sub.enum_members else ""
            print(f"      {sub.name:28s}  kind={sub.kind.value:10s}  type={sub_type}{members_note}")


## Stage 4 — Inspect Free vs Fixed Slots

This is what `LLMBackend._evaluate()` does internally to split the Match expression into free slots (LLM must fill) and fixed slots (already provided by the user).

In [ ]:
def prompt_slot_name(name, action_cls):
    prefix = f"{action_cls.__name__}."
    return name[len(prefix):] if name.startswith(prefix) else name

def match_bindings(match):
    bindings = []
    for attr in match.matches_with_variables:
        variable = attr.assigned_variable
        full_name = attr.name_from_variable_access_path
        value = getattr(variable, "_value_", None)
        bindings.append({
            "attribute_name": attr.attribute_name,
            "prompt_name": prompt_slot_name(full_name, match.type),
            "field_type": variable._type_,
            "value": value,
            "is_free": isinstance(value, type(Ellipsis)),
        })
    return bindings

bindings = match_bindings(match)
free_bindings = [binding for binding in bindings if binding["is_free"]]
fixed_slots = {
    binding["prompt_name"]: binding["value"]
    for binding in bindings
    if not binding["is_free"]
}
free_slots = [(binding["prompt_name"], binding["field_type"]) for binding in free_bindings]

print("FREE slots  (LLM will fill these):")
for name, typ in free_slots:
    print(f"  {name:45s}  type: {typ}")

print("FIXED slots (already known — passed as context to LLM):")
if fixed_slots:
    for name, val in fixed_slots.items():
        print(f"  {name:45s}  =  {val!r}")
else:
    print("  (none — user left everything open)")


## Stage 4b — All Free Slots go to the LLM

All free slots — including `Manipulator` and `Camera` fields — are sent to the slot filler. They are Symbol subclasses and are grounded from the SymbolGraph like any other entity (`Body`, `Region`, etc.).

There is no longer a separate "context injection" step.


In [ ]:
llm_free_slot_names = [binding["prompt_name"] for binding in free_bindings]

def field_spec_for_slot(name):
    if name in field_specs:
        return field_specs[name]
    if "." not in name:
        return None
    parent_name, sub_name = name.split(".", 1)
    parent = field_specs.get(parent_name)
    if parent is None:
        return None
    for sub in parent.sub_fields:
        if sub.name == sub_name:
            return sub
    return None

print("LLM-fillable fields (all free slots):")
for name in llm_free_slot_names:
    fspec = field_spec_for_slot(name)
    kind_str = fspec.kind.value if fspec else "unknown"
    print(f"  {name:45s}  kind={kind_str}")

print()
print("Entity slots are grounded from SymbolGraph; enum and primitive slots are coerced after the LLM returns strings.")


## Stage 5 — Serialize World via SymbolGraph

`render_world_context(symbol_type)` builds the LLM world-context string from `SymbolGraph` — no world object is passed.

The serializer now emits a structured grounding-oriented snapshot: summary, grounding instructions, scene objects, semantic types, spatial context, and symbol relations. Robot kinematic structures are filtered semantically from robot annotations when available, with the old name-suffix filter only as a fallback.


In [ ]:
from llmr.bridge.world_reader import WorldContextConfig, render_world_context

world_context = render_world_context(
    symbol_type,
    options=WorldContextConfig(
        max_objects=117,
        max_relations=0,
        include_geometry=False,
        include_parent_context=True,
        include_relations=False,
        exclude_robot_structures=True
    ),
)
print(world_context)


## Stage 6a — Inspect Dynamic Prompt

`fill_slots()` builds this prompt internally. This optional inspection cell uses private prompt helpers only so you can see the exact message before the LLM call; normal user code should call `fill_slots()`, `instance_from_match()`, `plan_from_match()`, or `plan_from_instruction()`.


## Stage 6a — Dynamic Prompt Boundary

`fill_slots()` builds the action-specific prompt internally from the action schema, free slot names, fixed slot values, and serialized world context.

The notebook intentionally avoids importing private prompt helpers. The next cell calls the public `fill_slots()` entry point and prints the structured `SlotFillingOutput`.

## Stage 6b — LLM Slot Filling (Core Reasoning)

`fill_slots()` sends the dynamic prompt to the LLM and returns an `SlotFillingOutput` — a structured list of `SlotValue` objects.

For entity slots: each `SlotValue` carries an `EntityDescription` (name + semantic_type + spatial_context + attributes) used by `EntityGrounding` in the next stage.

For enum/primitive slots: `SlotValue.value` is the resolved string (e.g. `"LEFT"`, `"TOP"`).

In [ ]:
from llmr.reasoning.slot_filler import fill_slots
from llmr.schemas import SlotFillingOutput

output: SlotFillingOutput = fill_slots(
    instruction     = INSTRUCTION,
    action_cls      = action_cls,
    free_slot_names = llm_free_slot_names,
    fixed_slots     = fixed_slots,
    world_context   = world_context,
    llm             = llm,
)

assert output is not None, "Slot filler returned None — check LLM connectivity"

print(f"=== SlotFillingOutput ===")
print(f"action_type      : {output.action_type}")
print(f"overall_reasoning: {output.overall_reasoning}")
print(f"\nSlots ({len(output.slots)}):")

for sv in output.slots:
    print(f"\n  field_name : {sv.field_name}")
    print(f"  value      : {sv.value!r}")
    if sv.entity_description:
        ed = sv.entity_description
        print(f"  entity_description:")
        print(f"    name            = {ed.name!r}")
        print(f"    semantic_type   = {ed.semantic_type!r}")
        print(f"    spatial_context = {ed.spatial_context!r}")
        print(f"    attributes      = {ed.attributes}")
    print(f"  reasoning  : {sv.reasoning}")


## Stage 7 — Entity Grounding via EntityGrounding

For ENTITY/POSE slots, `LLMBackend` calls `EntityGrounding.ground(entity_description)` which runs a two-tier search in `SymbolGraph`:

- Tier 1: annotation-based lookup from `semantic_type` to SymbolGraph instances
- Tier 2: name-based fallback over instances of `symbol_type`

The LLM sees names and semantic descriptions. Deterministic grounding happens here.


In [ ]:
from llmr.resolution.grounder import EntityGrounding
from llmr.bridge.introspect import FieldKind
from llmr.schemas import EntityDescription

grounder = EntityGrounding(symbol_type)
slot_by_name = {sv.field_name: sv for sv in output.slots}

print("=== Entity grounding results ===")

for fspec in action_schema.fields:
    if fspec.kind not in (FieldKind.ENTITY, FieldKind.POSE, FieldKind.TYPE_REF):
        continue
    sv = slot_by_name.get(fspec.name)
    if sv is None:
        print(f"  {fspec.name}: no SlotValue from LLM")
        continue

    grounding_ed = sv.entity_description or EntityDescription(name=sv.value or fspec.name)
    result = grounder.ground(grounding_ed)
    print(f"  {fspec.name}:")
    print(f"    EntityDescription: name={grounding_ed.name!r}  semantic_type={grounding_ed.semantic_type!r}")
    if result.warning:
        print(f"    warning: {result.warning}")
    if result.candidates:
        from llmr.bridge.world_reader import symbol_display_name
        print(f"    grounded to: {[symbol_display_name(b) for b in result.candidates]}")
        print(f"    using first: {symbol_display_name(result.candidates[0])}")
    else:
        print("    grounded to: NOTHING (entity not found in world)")


## Stage 8 — Write Resolved Values into Match → Construct Instance

This cell intentionally mirrors the core resolution loop inside `LLMBackend._evaluate()` so you can inspect the final action instance produced from an already-resolved LLM `SlotFillingOutput`.

This is a debugging/testing view of the backend internals. For normal usage, use `instance_from_match()` in the next stage.


In [ ]:
# world.semantic_annotations[6]

In [ ]:
from llmr import instance_from_match

# Build a fresh Match so this manual test is repeatable even if earlier cells mutated `match`.
manual_match = underspecified_match_from_schema(action_cls, action_schema)

print("=== Before instance_from_match — Match variables ===")
for attr in manual_match.matches_with_variables:
    name = prompt_slot_name(attr.name_from_variable_access_path, manual_match.type)
    print(f"  {name:45s}  _value_ = {attr.assigned_variable._value_!r}")

action_instance = instance_from_match(
    match=manual_match,
    llm=llm,
    symbol_type=symbol_type,
    instruction=INSTRUCTION,
    strict_required=True,
)

print("=== Constructed action instance through public instance_from_match() ===")
print(f"  Type : {type(action_instance).__name__}")
print(f"  Value: {action_instance}")


In [ ]:
print(type(action_instance))
print(action_instance.__dict__.keys())
print(action_instance.object_designator, type(action_instance.object_designator))
print(action_instance.grasp_description.manipulator.name)

## Execution

In [ ]:
from pycram.datastructures.enums import Arms
from pycram.locations.locations import CostmapLocation
from pycram.motion_executor import simulated_robot
from pycram.plans.factories import sequential
from pycram.robot_plans.actions.core.navigation import NavigateAction
from pycram.robot_plans.actions.core.pick_up import PickUpAction
from pycram.robot_plans.actions.core.robot_body import ParkArmsAction, MoveTorsoAction
from semantic_digital_twin.datastructures.definitions import TorsoState

# Use the object already resolved by llmr.
object_designator = action_instance.object_designator

# Put ROBOT in a task-ready posture before generating the costmap pose.
setup_plan = sequential(
    [
        ParkArmsAction(Arms.BOTH),
        MoveTorsoAction(TorsoState.HIGH),
    ],
    context=context,
)

with simulated_robot:
    setup_plan.perform()


In [ ]:
plan_node = sequential(
    [
        NavigateAction(CostmapLocation(target=action_instance.object_designator.global_pose,
                                       reachable=True,
                                       reachable_arm=action_instance.arm,
                                       grasp_description=action_instance.grasp_description,
                                       context=context).ground(), keep_joint_states=True),
        PickUpAction(
            object_designator=action_instance.object_designator,
            arm=action_instance.arm,
            grasp_description=action_instance.grasp_description,
        ),
    ],
    context=context,
)

print("Navigate + PickUp plan ready:", plan_node)


In [ ]:
with simulated_robot:
    plan_node.perform()


## Stage 9a — Resolve Parameters into an Action Instance

Use the public `instance_from_match()` API when you already have a `Match` and want the concrete action instance without creating or executing a `PlanNode`.

Internally this uses `LLMBackend`, resolves leaf slots from the LLM, grounds entities with `EntityGrounding` or SymbolGraph auto-grounding, writes values into the `Match`, and lets KRROOD construct nested dataclass instances.


In [ ]:
# Build a fresh Match so this stage is independent of any previous mutations.
params_match = underspecified_match_from_schema(action_cls, action_schema)

print("Params Match slots:")
for attr in params_match.matches_with_variables:
    print(f"  {attr.name_from_variable_access_path:45s}  {attr.assigned_variable._value_!r}")


In [ ]:
# params_match.__dict__

In [ ]:
from llmr import instance_from_match

action_instance = instance_from_match(
    match=params_match,
    llm=llm,
    symbol_type=symbol_type,
    instruction=INSTRUCTION,
    strict_required=True,
)

print("=== Constructed action instance ===")
print(f"  Type : {type(action_instance).__name__}")
# print(f"  Value: {action_instance}")


In [ ]:
# action_cls

In [ ]:
print(type(action_instance))
print(action_instance.__dict__.keys())

In [ ]:
print(action_instance.object_designator, type(action_instance.object_designator))
print(action_instance.arm, type(action_instance.arm))
print(action_instance.grasp_description.__dict__.keys(), type(action_instance.grasp_description))
print(action_instance.grasp_description.approach_direction, type(action_instance.grasp_description.approach_direction))
print(action_instance.grasp_description.vertical_alignment, type(action_instance.grasp_description.vertical_alignment))

In [ ]:
# action_instance.grasp_description.manipulator

## Stage 9b— Role 2: `plan_from_match()` Plan Node Resolver

`plan_from_match()` is the Role 2 entry point when the action type is already known and you want a PyCRAM `PlanNode` for an underspecified `Match`.

Use `instance_from_match()` instead when you want only the concrete action instance without plan-node construction.


In [ ]:
from krrood.entity_query_language.query.match import Match
from llmr import plan_from_match
from pycram.datastructures.enums import Arms
from pycram.datastructures.grasp import GraspDescription

# Pre-fix arm=RIGHT — LLM fills object_designator and grasp-description leaves.
power_match = Match(action_cls)(
    object_designator=...,        # free — LLM grounds from world context
    arm=Arms.RIGHT,               # fixed — forced by caller
    grasp_description=Match(GraspDescription)(
        approach_direction=...,
        vertical_alignment=...,
        manipulator=...,
    ),
)

print("Match slots:")
for attr in power_match.matches_with_variables:
    name = attr.name_from_variable_access_path
    value = attr.assigned_variable._value_
    status = "FREE (LLM fills)" if isinstance(value, type(Ellipsis)) else f"FIXED = {value!r}"
    print(f"  {name:45s}  {status}")

plan_node = plan_from_match(
    match=power_match,
    context=context,
    llm=llm,
    symbol_type=symbol_type,
    instruction=INSTRUCTION,
    strict_required=True,
)

print()
print(f"Plan node  : {plan_node}")
print(f"Backend    : {type(context.query_backend).__name__}")
print(f"groundable : {context.query_backend.symbol_type.__name__}")


## Stage 10 — Simple User API: `plan_from_instruction()` end-to-end

`plan_from_instruction()` runs all previous stages automatically. The user provides only the NL instruction and `symbol_type`. No Match, no action class, no backend — just language.

In [ ]:
from llmr import plan_from_instruction

# symbol_type is optional (defaults to Symbol).
# Pass Body for physical-body-scoped grounding.
plan_node = plan_from_instruction(
    instruction     = INSTRUCTION,
    context         = context,
    llm             = llm,
    symbol_type = symbol_type,
)

print(f"Plan node  : {plan_node}")
print(f"Backend    : {type(context.query_backend).__name__}")
print(f"Instruction: {context.query_backend.instruction!r}")
print(f"groundable : {context.query_backend.symbol_type.__name__}")


## Stage 10b — Navigate Before Pickup

A bare `PickUpAction` only checks whether the object is reachable from the robot's current base pose. If every arm/grasp candidate is unreachable, use PyCRAM's `CostmapLocation` to generate a reachable base pose near the object, then execute `NavigateAction` before `PickUpAction`.

Important: posture setup and final execution must run inside PyCRAM's `simulated_robot` execution environment. If you call `.perform()` without it, PyCRAM logs `Unknown execution type: None`, and the robot state will not be updated for reachability checks.


In [ ]:
from pycram.datastructures.enums import Arms
from pycram.locations.locations import CostmapLocation
from pycram.motion_executor import simulated_robot
from pycram.plans.factories import sequential
from pycram.robot_plans.actions.core.navigation import NavigateAction
from pycram.robot_plans.actions.core.pick_up import PickUpAction
from pycram.robot_plans.actions.core.robot_body import ParkArmsAction, MoveTorsoAction
from semantic_digital_twin.datastructures.definitions import TorsoState

# Use the object already resolved by llmr.
object_designator = action_instance.object_designator

# Put PR2 in a reach-friendly posture before generating the costmap pose.
setup_plan = sequential(
    [
        ParkArmsAction(Arms.BOTH),
        MoveTorsoAction(TorsoState.HIGH),
    ],
    context=context,
)

with simulated_robot:
    setup_plan.perform()


In [ ]:
action_instance.arm

In [ ]:
# pickup_pose = None
# pickup_arm = None
#
# for arm in (Arms.LEFT, Arms.RIGHT):
#     location = CostmapLocation(
#         target=object_designator.global_pose,
#         reachable=True,
#         reachable_arm=arm,
#         context=context,
#     )
#     try:
#         candidate = location.ground()
#     except StopIteration:
#         candidate = None
#
#     if candidate is None:
#         print(f"No reachable navigation pose found for {arm.name}")
#         continue
#
#     pickup_pose = candidate
#     pickup_arm = candidate.arm or arm
#     grasp = pickup_pose.grasp_description
#     print(f"Reachable pickup pose found with {pickup_arm.name}")
#     print("Base pose:", pickup_pose)
#     print("Generated grasp:", grasp.approach_direction, grasp.vertical_alignment)
#     break
#
# assert pickup_pose is not None, "PyCRAM could not find a reachable base pose for pickup"


In [ ]:
# Build a concrete navigate-then-pick plan. This bypasses the LLM for navigation;
# llmr is still used for object grounding, and PyCRAM supplies the feasible base pose/grasp.
plan_node = sequential(
    [
        NavigateAction(CostmapLocation(target=action_instance.object_designator.global_pose,
                                       reachable=True,
                                       reachable_arm=action_instance.arm,
                                       context=context).ground(), keep_joint_states=True),
        PickUpAction(
            object_designator=action_instance.object_designator,
            arm=action_instance.arm,
            grasp_description=action_instance.grasp_description,
        ),
    ],
    context=context,
)

print("Navigate + PickUp plan ready:", plan_node)


## Stage 11 — Execute Navigate + Pickup

This executes the concrete `NavigateAction` followed by `PickUpAction`. Run it inside `simulated_robot`; otherwise PyCRAM will not have an execution backend for motion actions.


In [ ]:
with simulated_robot:
    plan_node.perform()


## Bonus — Multi-Step: `sequential_plan_from_instruction()`

Tests `TaskDecomposer` + `plan_from_instruction()` chained together. Each step gets its own `LLMBackend` with a step-specific instruction and the shared `symbol_type`.

In [ ]:
from llmr.reasoning.decomposer import TaskDecomposer

MULTI_INSTRUCTION = "go to the table, pick up the milk, and put it in the fridge"

decomposer = TaskDecomposer(llm=llm)
decomposed = decomposer.decompose(MULTI_INSTRUCTION)

print("=== Decomposed steps ===")
for i, step in enumerate(decomposed.steps):
    deps = decomposed.dependencies.get(i, [])
    dep_str = f"  (depends on steps {deps})" if deps else ""
    print(f"  [{i}] {step}{dep_str}")

# Expected:
# [0] navigate to the table
# [1] pick up the milk from the table  (depends on steps [0])
# [2] place the milk in the fridge     (depends on steps [1])

In [ ]:
from llmr import sequential_plan_from_instruction

plan_nodes = sequential_plan_from_instruction(
    instruction     = MULTI_INSTRUCTION,
    context         = context,
    llm             = llm,
    symbol_type = symbol_type,
)

print(f"Built {len(plan_nodes)} plan nodes:")
for i, node in enumerate(plan_nodes):
    print(f"  [{i}] {node}")

print("\nExecuting...")
for i, node in enumerate(plan_nodes):
    print(f"\n--- Step {i} ---")
    node.perform()
